In [15]:
# Part 1- Basic Text Summarisation Using PromptTemplate
from langchain_classic.prompts import PromptTemplate
from langchain_community.document_loaders import PyPDFLoader

docs = PyPDFLoader('./paper.pdf').load()
print(sum(len(i.page_content) for i in docs))
print(docs[1].page_content)



13818
Research Papers in the Sciences, Fall 2019.   2 of 7 
 
Introduction 	
The introduction explains to the reader the context of the experiment and the basic goals, 
methods, and results. Context is needed to explain the overall goal and to give necessary 
background information: the topic, the relevant existing research done on the topic, and the 
connection the experiment has with past research (i.e., how the research is unique and 
important). In addition, the introduction should explain what hypothesis the experiment was 
intended to prove, what general method was used, and what the expected results were. It should 
provide a brief overview of the rest of the paper.  
 
Abridged Introduction Example  
 
Copper can be oxidized, meaning copper atoms react with the oxygen in air and form copper 
oxide. This leads to a change in properties, which can affect the structural integrity and safety of 
items that are made out of copper. A safe and inexpensive method that was hypothesized 

In [31]:
#Task 2
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
load_dotenv()
LLM = ChatGoogleGenerativeAI(model='gemini-3-flash-preview')

prompt = PromptTemplate(template='You are the summarizer ai which summarize the given text into short summary in 100 words. Summarize this {text}',
                        input_variables=['text'])
chain = prompt | LLM
res = chain.invoke({'text':docs})
print(res.content[0].get('text'))

This guide for undergraduate researchers outlines the essential components of a scientific paper: Abstract, Introduction, Materials and Methods, Results, and Discussion/Conclusion. It emphasizes three core principles: clarity for general audiences, sufficient detail for reproducibility, and contextualizing findings within existing research.

To maintain professional credibility, authors must use formal, objective language and a third-person perspective. The text advises using passive voice to avoid mentioning researchers explicitly, focusing instead on the experiment itself. By separating objective facts in the results from interpretations in the discussion, scientists effectively communicate how their work contributes to the broader scientific community.


In [33]:
prompt = PromptTemplate(template='You are the summarizer ai which summarize the given text into short summary in Short Summary(5-6 lines). Summarize this {text}',
                        input_variables=['text'])
prompt2 = PromptTemplate(template='You are the summarizer ai which summarize the given text into short summary in Bullet point summary. Summarize this {text}',
                        input_variables=['text'])
chain2 = prompt2 | LLM
res = chain.invoke({'text':docs})
res2 = chain2.invoke({'text':docs})
print(res.content[0].get('text'))
print()
print(res2.content[0].get('text'))

This guide outlines the essential components and stylistic requirements for writing undergraduate scientific research papers. To effectively communicate findings, papers must be clear, replicable, and contextualized within existing research. Key sections include the abstract, introduction, materials and methods, results, and a combined discussion and conclusion. 

The text emphasizes a formal, objective tone, traditionally using the third person and passive voice to prioritize the experiment over the researcher. By following these structural guidelines and focusing on objective data reporting, students can produce credible research that highlights the significance of their work while allowing for scientific validation.

This document from the San José State University Writing Center provides a comprehensive guide for undergraduate students on how to write a scientific research paper.

### **Core Principles of Scientific Writing**
*   **Understandability:** Papers must be accessible to 

Task 4: Why Stuff Chain is Needed (Conceptual) Answer briefly: 
1. What is a stuff summarization chain? 
Stuff summarization chain summarize all the document or chunks at once with the help of LLM.
2. When is it suitable to use?
For short document it is suitable. 
3. Limitations of stuff chain.
For long document the context length will not be enough for the processing and memory will not be efficiently used of the LLM

In [ ]:
# Task 5
from langchain_classic.chains.summarize import load_summarize_chain
chain = load_summarize_chain(llm=LLM,chain_type='stuff',prompt=prompt,verbose=True)
res = chain.run(docs)
res



> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
You are the summarizer ai which summarize the given text into short summary in Short Summary(5-6 lines). Summarize this San José State University Writing Center 
www.sjsu.edu/writingcenter  
Written by Jane Juan  
Research Papers in the Sciences, Fall 2019.   1 of 7 
 
Research Papers in the Sciences (Undergraduate)    
 
Scientific research is shared through scientific research papers. It’s considered part of the duty of 
the scientist to share information with the scientific community. There are three important things 
to keep in mind when writing a scientific research paper as an undergraduate researcher. First, 
the paper should be written understandably for an audience that may know nothing about the 
research. Second, the paper should be written in enough detail so that the experiment can be 
replicated to show the legitimacy of the findings. Finally, it’s important to know w

'Scientific research papers serve to share reproducible findings with the community by providing clear context and detailed methodology. They are structured into standard sections: an abstract for summarization, an introduction for background, a narrative-style methods section, and an objective results section. The final discussion and conclusion interpret the data, address the hypothesis, and note potential errors or future applications. To maintain professional credibility, authors must use a formal, objective tone, typically favoring a third-person perspective and avoiding personal mentions. Ultimately, the goal is to present information so clearly that the research can be understood and replicated by other scientists.'

Task 6: Comparison with Prompt-Based Summary!!!
1.Compare: Prompt-only summary Stuff chain summary?
Both have same function stuff chain is just a wrapper that help in reducing the length of the code.
2.Discuss differences in output quality and limitations.
Both will have same output. and both limitations are same i.e ' Not suitable for long Documents'

Part 3 - Map Reduce Summarization
Task 7: Why Map-Reduce is Needed (Conceptual) Answer briefly: 
1. Why large documents need map-reduce summarization?
For LLM to process the large documents it is important to do map-reduce summarization since it preserves the context window of the LLM and helps to get all information which out any loss.
2. How map and reduce steps work.
It works on chunks, summarizing all the chunks seperately then combining all the summary into one.

In [40]:
# Task 8- Implement Map Reduce Summarization Chain
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq
splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=0)
chunks = splitter.split_documents(docs)
LLM = ChatGroq(model='openai/gpt-oss-20b')
chain = load_summarize_chain(llm=LLM,chain_type='map_reduce',map_prompt=prompt,combine_prompt=prompt,verbose=True)
res = chain.run(chunks)




> Entering new MapReduceDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
You are the summarizer ai which summarize the given text into short summary in Short Summary(5-6 lines). Summarize this San José State University Writing Center 
www.sjsu.edu/writingcenter  
Written by Jane Juan  
Research Papers in the Sciences, Fall 2019.   1 of 7 
 
Research Papers in the Sciences (Undergraduate)    
 
Scientific research is shared through scientific research papers. It’s considered part of the duty of 
the scientist to share information with the scientific community. There are three important things 
to keep in mind when writing a scientific research paper as an undergraduate researcher. First, 
the paper should be written understandably for an audience that may know nothing about the 
research. Second, the paper should be written in enough detail so that the experiment can be 
replicated to show the legitimacy of the findings. Finally, it’s important to kn

d:\miniconda3\envs\genai\Lib\site-packages\langchain_core\language_models\base.py:463: UserWarning: Using fallback GPT-2 tokenizer for token counting. Token counts may be inaccurate for non-GPT-2 models. For accurate counts, use a model-specific method if available.
  return len(self.get_token_ids(text))
d:\miniconda3\envs\genai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm




> Entering new LLMChain chain...
Prompt after formatting:
You are the summarizer ai which summarize the given text into short summary in Short Summary(5-6 lines). Summarize this **Short Summary**

San José State University’s Writing Center offers guidance for undergraduate scientists on crafting research papers. The guide stresses that papers must be accessible to readers who may not be familiar with the topic, ensuring clarity and comprehensibility. It also emphasizes the need for detailed methodology so experiments can be replicated, thereby validating the findings. Finally, authors are urged to contextualize their work within existing scientific literature, demonstrating how their research builds on or differs from current studies.

**Short Summary**

Scientific papers must be organized into clear, titled sections—most importantly an abstract that quickly informs readers of the study’s purpose, methods, results, and conclusions. The abstract (150–250 words) should be concise yet c

In [42]:
res

'**Short Summary**\n\nThe passage presents a series of model “Short Summary” excerpts illustrating how to condense scientific content into 5‑6 concise sentences. It covers key elements of scientific writing—clarity for non‑experts, detailed methodology, contextualization within literature, and structured sections such as abstract, methods, results, and discussion. Examples include a copper‑oxidation removal experiment with vinegar, guidelines for writing materials‑and‑methods and results sections, and advice on point‑of‑view and voice. The material serves as a template for drafting objective, reproducible, and reader‑friendly summaries in research papers.'

PART 4 - Refine Summarization Chain  
Task 10 :Understanding Refine Chain (Conceptual) Answer briefly: 
1. How refine summarization works?
It works like first it summarize the chunk1 then with summary 1 + chuck2 summary 2 is generated like this loop works and final summary is generated. 
2. Difference between map-reduce and refine?
map-reduce - each summary is independent i.e no relation between summary 1 and 2,
while in refine it's dependent.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size=5000,chunk_overlap=0)
chunks = splitter.split_documents(docs)
LLM = ChatGroq(model='openai/gpt-oss-20b')
question_prompt = PromptTemplate(template='write the concise summary of the text: {text}',
                                 input_variables=['text'])
refine_prompt = PromptTemplate(template='Take this summary : {existing_answer} , and generate the addition summary for this text: {text}',
                               input_variables=['existing_answer','text'])
chain = load_summarize_chain(llm=LLM,chain_type='refine',question_prompt=question_prompt,refine_prompt=refine_prompt,verbose=True)
res = chain.run(chunks)




> Entering new RefineDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
write the concise summary of the text: San José State University Writing Center 
www.sjsu.edu/writingcenter  
Written by Jane Juan  
Research Papers in the Sciences, Fall 2019.   1 of 7 
 
Research Papers in the Sciences (Undergraduate)    
 
Scientific research is shared through scientific research papers. It’s considered part of the duty of 
the scientist to share information with the scientific community. There are three important things 
to keep in mind when writing a scientific research paper as an undergraduate researcher. First, 
the paper should be written understandably for an audience that may know nothing about the 
research. Second, the paper should be written in enough detail so that the experiment can be 
replicated to show the legitimacy of the findings. Finally, it’s important to know what else is 
being discussed on your topic in the scientific community and to th

Task 12: Comparison of All Summarization Methods ,
Compare: 
Prompt-based 
Stuff chain 
Map-reduce chain 
Refine chain  
Evaluate based on: 
Summary quality 
Coherence 
Suitability for long documents


Ans:- 

Prompt-based & Stuff Chain
Both offer high summary quality and high coherence because the LLM sees the entire text at once. However, they are very poor for long documents—if the text exceeds the context window, the process will crash. (The only difference is that Stuff automates the string concatenation).

Map-Reduce Chain
Offers moderate summary quality and moderate coherence because it summarizes chunks individually and then summarizes those summaries (losing some nuance). However, it is excellent for long documents, as it processes chunks in parallel without hitting token limits.

Refine Chain
Offers high summary quality and high coherence because it sequentially builds on the previous summary chunk-by-chunk. It is good for long documents, but it is much slower and more expensive than Map-Reduce.

In [62]:
# Task 13: Build a Summarization Function:
def summarize_document(text,method='map_reduce'):
    doc = text
    summary = 'None'
    splitter = RecursiveCharacterTextSplitter(chunk_size=3000,chunk_overlap=0)
    chunks = splitter.split_documents(doc)
    if method == 'stuff':
        prompt = PromptTemplate(template='You are the summarizer ai which summarize the given text into short summary in 100 words. Summarize this {text}',
                        input_variables=['text'])
        chain = load_summarize_chain(llm=LLM,chain_type='stuff',prompt=prompt,verbose=True)
    elif method == 'map_reduce':
        prompt = PromptTemplate(template='You are the summarizer ai which summarize the given text into short summary in 100 words. Summarize this {text}',
                                input_variables=['text'])
        chain = load_summarize_chain(llm=LLM,chain_type='map_reduce',map_prompt=prompt,combine_prompt=prompt,verbose=True)
    elif method == 'refine':
        question_prompt = PromptTemplate(template='write the concise summary of the text: {text}',
                                input_variables=['text'])
        refine_prompt = PromptTemplate(template='Take this summary : {existing_answer} , and generate the addition summary for this text: {text}',
                                    input_variables=['existing_answer','text'])
        chain = load_summarize_chain(llm=LLM,chain_type='refine',question_prompt=question_prompt,refine_prompt=refine_prompt,verbose=True)
    summary = chain.run(chunks)

    return summary

In [65]:
summarize_document(docs,'map_reduce')



> Entering new MapReduceDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
You are the summarizer ai which summarize the given text into short summary in 100 words. Summarize this San José State University Writing Center 
www.sjsu.edu/writingcenter  
Written by Jane Juan  
Research Papers in the Sciences, Fall 2019.   1 of 7 
 
Research Papers in the Sciences (Undergraduate)    
 
Scientific research is shared through scientific research papers. It’s considered part of the duty of 
the scientist to share information with the scientific community. There are three important things 
to keep in mind when writing a scientific research paper as an undergraduate researcher. First, 
the paper should be written understandably for an audience that may know nothing about the 
research. Second, the paper should be written in enough detail so that the experiment can be 
replicated to show the legitimacy of the findings. Finally, it’s important to know what else is

'The handout outlines essential elements for clear, reproducible rigorous undergraduate research papers. It stresses that each section—Abstract, Introduction, Materials & Methods, Results, Discussion, Conclusion—must contain specific content, with the abstract (150‑250 words) summarizing purpose, methods, findings, significance. Guidelines from the San\u202fJosé State Writing Center emphasize proper citations and objective language, avoiding first‑person pronouns and explicit researcher names. An example study on copper oxidation demonstrates removing the oxidized layer with vinegar, detailing experimental steps, results, and potential future work on acetic‑acid concentration. The guide also covers writing style, passive/active voice choices, and structuring discussion and conclusion to link data to existing literature.'

Task 14: Observations & Insights Write short answers: 
1. Best summarization strategy for very long documents?

Refine Summarization is best for very long documents as it preserves the context

2. Trade-offs between speed and quality?

Speed is good in Stuff and the quality is good in Stuff(for short docs) and Refine(for long docs)

3. Real-world use cases of each method

Stuff is used for normal text summarization,

Map Reduce is used for text summarization where the context is not that important and we want fast result,

Refine is used for research paper summarization